# DATE Calculation 

### DATEADD()

#### Adds or subtracts a specific time interval to/from a date

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_customers = pd.read_csv('data/sales_customers.csv')
df_employees = pd.read_csv('data/sales_employees.csv')
df_orders = pd.read_csv('data/sales_orders.csv')
df_orderarchive = pd.read_csv('data/sales_ordersarchive.csv')
df_products = pd.read_csv('data/sales_products.csv')

```SQL
SELECT
    orderid,
    orderdate,
    (orderdate + INTERVAL  '2 years 3 months -10 days')::DATE AS orderdate_plus,
    (orderdate + INTERVAL  '2 years')::DATE AS orderdate_plus_years,
    (orderdate + INTERVAL  '3 months')::DATE AS orderdate_plus_month,
    (orderdate + INTERVAL  '-10 days')::DATE AS orderdate_plus_days
FROM sales.orders;
```

In [3]:

df_or = df_orders.copy()

# 1. Aseguramos que orderdate sea datetime
df_or['orderdate'] = pd.to_datetime(df_or['orderdate'])

# 2. Aplicamos los intervalos
# Para el caso complejo (años, meses y días a la vez)
df_or['orderdate_plus'] = (df_or['orderdate'] + pd.DateOffset(years=2, months=3, days=-10)).dt.date

# Casos individuales
df_or['orderdate_plus_years'] = (df_or['orderdate'] + pd.DateOffset(years=2)).dt.date
df_or['orderdate_plus_month'] = (df_or['orderdate'] + pd.DateOffset(months=3)).dt.date
df_or['orderdate_plus_days']  = (df_or['orderdate'] + pd.DateOffset(days=-10)).dt.date

df_or

,orderid,productid,customerid,salespersonid,orderdate,shipdate,orderstatus,shipaddress,billaddress,quantity,sales,creationtime,orderdate_plus,orderdate_plus_years,orderdate_plus_month,orderdate_plus_days
0,1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,10,2025-01-01 12:34:56.000000,2027-03-22,2027-01-01,2025-04-01,2024-12-22
1,2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,NaN,1,15,2025-01-05 23:22:04.000000,2027-03-26,2027-01-05,2025-04-05,2024-12-26
2,3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,20,2025-01-10 18:24:08.000000,2027-03-31,2027-01-10,2025-04-10,2024-12-31
3,4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,NaN,2,60,2025-01-20 05:50:33.000000,2027-04-10,2027-01-20,2025-04-20,2025-01-10
4,5,104,2,5,2025-02-01,2025-02-05,Delivered,NaN,NaN,1,25,2025-02-01 14:02:41.000000,2027-04-21,2027-02-01,2025-05-01,2025-01-22
5,6,104,3,5,2025-02-05,2025-02-10,Delivered,1792 Belmont Rd.,NaN,2,50,2025-02-06 15:34:57.000000,2027-04-25,2027-02-05,2025-05-05,2025-01-26
6,7,102,1,1,2025-02-15,2025-02-27,Delivered,136 Balboa Court,NaN,2,30,2025-02-16 06:22:01.000000,2027-05-05,2027-02-15,2025-05-15,2025-02-05
7,8,101,4,3,2025-02-18,2025-02-27,Shipped,2947 Vine Lane,4311 Clay Rd,3,90,2025-02-18 10:45:22.000000,2027-05-08,2027-02-18,2025-05-18,2025-02-08
8,9,101,2,3,2025-03-10,2025-03-15,Shipped,3768 Door Way,NaN,2,20,2025-03-10 12:59:04.000000,2027-05-31,2027-03-10,2025-06-10,2025-02-28
9,10,102,3,5,2025-03-15,2025-03-20,Shipped,NaN,NaN,0,60,2025-03-16 23:25:15.000000,2027-06-05,2027-03-15,2025-06-15,2025-03-05


### DATEDIFF

### SQL TASK

#### Calculate the age of employees

```SQL
SELECT
    employeeid,
    birthdate,
    EXTRACT(YEAR FROM AGE(NOW(), birthdate)) AS age
FROM sales.employees;
```

In [4]:
df_age = df_employees.copy()

df_age['birthdate'] = pd.to_datetime(df_age['birthdate'])

hoy = pd.Timestamp.now()

df_age['age'] = ((hoy - df_age['birthdate']).dt.days / 365.25).astype(int)

df_age




,employeeid,firstname,lastname,department,birthdate,gender,salary,managerid,age
0,1,Frank,Lee,Marketing,1988-12-05,M,55000,NaN,37
1,2,Kevin,Brown,Marketing,1972-11-25,M,65000,1.0,53
2,3,Mary,NaN,Sales,1986-01-05,F,75000,1.0,40
3,4,Michael,Ray,Sales,1977-02-10,M,90000,2.0,48
4,5,Carol,Baker,Sales,1982-02-11,F,55000,3.0,43


### SQL TASK

#### Find the average shipping duration in days for each month

```SQL
SELECT
    EXTRACT(MONTH FROM orderdate) AS orderdate,
    AVG((shipdate - orderdate)) AS diff_date
FROM sales.orders
GROUP BY EXTRACT(MONTH FROM orderdate) ;
```

In [7]:
df_month_or = df_orders.copy()

df_month_or['orderdate'] = pd.to_datetime(df_month_or['orderdate'])
df_month_or['shipdate'] = pd.to_datetime(df_month_or['shipdate'])

df_month_or['diff_date'] = (df_month_or['shipdate'] - df_month_or['orderdate']).dt.days
df_month_or['month'] = df_month_or['orderdate'].dt.month

result = df_month_or.groupby('month')['diff_date'].mean().reset_index()

result

,month,diff_date
0,1,7.25
1,2,7.50
2,3,5.00


### SQL TASK

#### Find the number of days between each order and previous order

```SQL
-- OPCIÓN 1
SELECT
    orderID,
    orderdate currentorderdate,
    LAG(orderdate) OVER (ORDER BY  orderdate) AS previousorderdate,
    orderdate - LAG(orderdate) OVER (ORDER BY orderdate) AS numberofdays
FROM sales.orders

-- OPCIÓN 2
WITH orders_with_lag AS (
    SELECT
        orderid,
        orderdate,
        LAG(orderdate) OVER (ORDER BY orderdate) as previousorderdate
    FROM sales.orders
)
SELECT
    orderid,
    orderdate AS currentorderdate,
    previousorderdate,
    orderdate - previousorderdate AS numberofdays
FROM orders_with_lag;
```

In [11]:
df_or = df_orders.copy()
df_or['orderdate'] = pd.to_datetime(df_or['orderdate'])

df_or = df_or.sort_values(by='orderdate').reset_index(drop=True)

df_or['previousorderdate'] = df_or['orderdate'].shift(1)

df_or['numberofdays'] = (df_or['orderdate'] - df_or['previousorderdate']).dt.days

df_or

,orderid,productid,customerid,salespersonid,orderdate,shipdate,orderstatus,shipaddress,billaddress,quantity,sales,creationtime,previousorderdate,numberofdays
0,1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,10,2025-01-01 12:34:56.000000,NaT,NaN
1,2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,NaN,1,15,2025-01-05 23:22:04.000000,2025-01-01,4.0
2,3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,20,2025-01-10 18:24:08.000000,2025-01-05,5.0
3,4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,NaN,2,60,2025-01-20 05:50:33.000000,2025-01-10,10.0
4,5,104,2,5,2025-02-01,2025-02-05,Delivered,NaN,NaN,1,25,2025-02-01 14:02:41.000000,2025-01-20,12.0
5,6,104,3,5,2025-02-05,2025-02-10,Delivered,1792 Belmont Rd.,NaN,2,50,2025-02-06 15:34:57.000000,2025-02-01,4.0
6,7,102,1,1,2025-02-15,2025-02-27,Delivered,136 Balboa Court,NaN,2,30,2025-02-16 06:22:01.000000,2025-02-05,10.0
7,8,101,4,3,2025-02-18,2025-02-27,Shipped,2947 Vine Lane,4311 Clay Rd,3,90,2025-02-18 10:45:22.000000,2025-02-15,3.0
8,9,101,2,3,2025-03-10,2025-03-15,Shipped,3768 Door Way,NaN,2,20,2025-03-10 12:59:04.000000,2025-02-18,20.0
9,10,102,3,5,2025-03-15,2025-03-20,Shipped,NaN,NaN,0,60,2025-03-16 23:25:15.000000,2025-03-10,5.0


### ISDATE ()

#### Check if a value is a date